In [6]:
# Install packages (Google Colab)
!pip install -q gradio transformers torch black autopep8 reportlab

import re
import ast
import gradio as gr
import autopep8
import black
from transformers import pipeline
from reportlab.pdfgen import canvas

# -----------------------------
# Chatbot Configuration
# -----------------------------
class ChatbotConfig:
    MODEL_NAME = "gpt2"   # changed to small model
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 100
    DEFAULT_TOP_P = 0.95


# -----------------------------
# Load Model
# -----------------------------
model = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME
)

conversation_history = []

# -----------------------------
# Detect Python Code
# -----------------------------
def detect_python_code(text):

    patterns = [
        r'def\s+\w+\s*\(',
        r'class\s+\w+',
        r'import\s+\w+',
        r'from\s+\w+\s+import',
        r'if\s+.*:',
        r'for\s+.*\s+in\s+',
        r'while\s+.*:'
    ]

    for p in patterns:
        if re.search(p, text):
            return True

    return False


# -----------------------------
# Check Python Syntax
# -----------------------------
def check_syntax(code):

    try:
        ast.parse(code)
        return True, "No syntax errors"

    except SyntaxError as e:
        return False, f"Syntax Error line {e.lineno}: {e.msg}"


# -----------------------------
# Fix Python Code
# -----------------------------
def fix_python_code(code):

    try:
        fixed = autopep8.fix_code(code)

        try:
            fixed = black.format_str(fixed, mode=black.FileMode())
        except:
            pass

        return fixed

    except:
        return code


# -----------------------------
# Generate AI Response
# -----------------------------
def generate_response(message, temperature, max_tokens, top_p):

    output = model(
        message,
        max_new_tokens=int(max_tokens),
        temperature=temperature,
        do_sample=True,
        top_p=top_p
    )

    reply = output[0]["generated_text"]

    return reply


# -----------------------------
# Main Chat Function
# -----------------------------
def chatbot(message, history, temperature, max_tokens, top_p):

    if detect_python_code(message):

        ok, err = check_syntax(message)

        if not ok:
            fixed = fix_python_code(message)
            history.append((message, f"Fixed Code:\n{fixed}\n\nError: {err}"))
            return history

    reply = generate_response(message, temperature, max_tokens, top_p)

    history.append((message, reply))

    return history


# -----------------------------
# Gradio Interface
# -----------------------------
with gr.Blocks() as demo:

    gr.Markdown("# Syntax Surgeon - AI Code Fixer")

    chatbot_ui = gr.Chatbot(height=400)

    msg = gr.Textbox(label="Enter Message")

    send = gr.Button("Send")

    temp = gr.Slider(0.1,1,value=0.7,label="Temperature")
    tokens = gr.Slider(50,200,value=100,label="Max Tokens")
    top_p = gr.Slider(0.1,1,value=0.95,label="Top-P")

    send.click(
        chatbot,
        inputs=[msg, chatbot_ui, temp, tokens, top_p],
        outputs=chatbot_ui
    )

demo.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 11.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/tmp/ipykernel_450/3538385618.py:132: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(height=400)
/tmp/ipykernel_450/3538385618.py:132: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff970b5963058d29fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
!pip install -q gradio transformers torch black autopep8 reportlab requests
import re
import ast
import torch
import gradio as gr
import black
import autopep8
from transformers import pipeline
from datetime import datetime
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95
    device = 0 if torch.cuda.is_available() else -1

model_pipe = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME,
    device=ChatbotConfig.device
)

print("Model loaded successfully!")

def detect_python_code(text):
    patterns = [
        r"def\s+\w+\s*\(",
        r"class\s+\w+",
        r"import\s+\w+",
        r"from\s+\w+\s+import",
        r"if\s+.*:",
        r"for\s+.*\s+in\s+",
        r"while\s+.*:"
    ]

    for pattern in patterns:
        if re.search(pattern, text):
            return True

    return False

def check_syntax(code):
    try:
        ast.parse(code)
        return True, None

    except SyntaxError as e:
        return False, f"Syntax Error (line {e.lineno}): {e.msg}"

def fix_code_indentation(code):
    lines = code.split("\n")
    fixed = []

    for line in lines:
        line = line.rstrip()
        line = line.replace("\t", "    ")
        fixed.append(line)

    return "\n".join(fixed)

def fix_code_brackets(code):
    brackets = {
        "(": ")",
        "[": "]",
        "{": "}"
    }

    stack = []

    for char in code:
        if char in brackets:
            stack.append(brackets[char])
        elif char in brackets.values():
            if stack and stack[-1] == char:
                stack.pop()

    while stack:
        code += stack.pop()

    return code

def format_code(code):
    try:
        formatted = black.format_str(code, mode=black.FileMode())
        return formatted

    except Exception:
        return autopep8.fix_code(code)

def fix_python_code(code):
    code = fix_code_indentation(code)
    code = fix_code_brackets(code)

    syntax_ok, error = check_syntax(code)

    if syntax_ok:
        code = format_code(code)
        return "✅ Code fixed and formatted:\n\n" + code
    else:
        return f"⚠️ {error}\n\nAttempted Fix:\n{code}"

conversation_history = []

def generate_response(message, temperature, max_tokens, top_p):
    if detect_python_code(message):
        return fix_python_code(message)

    conversation_history.append(
        {"role": "user", "content": message}
    )

    response = model_pipe(
        message,
        max_new_tokens=int(max_tokens),
        temperature=temperature,
        do_sample=True,
        top_p=top_p
    )

    answer = response[0]["generated_text"]

    conversation_history.append(
        {"role": "assistant", "content": answer}
    )

    return answer

def download_txt():
    filename = "chat_history.txt"

    with open(filename, "w") as f:
        for msg in conversation_history:
            role = msg["role"].upper()
            f.write(f"{role}: {msg['content']}\n\n")

    return filename

def download_pdf():
    filename = "chat_history.pdf"

    styles = getSampleStyleSheet()
    story = []

    for msg in conversation_history:
        role = msg["role"].upper()
        text = f"{role}: {msg['content']}"
        story.append(Paragraph(text, styles["Normal"]))
        story.append(Spacer(1,10))

    doc = SimpleDocTemplate(filename)
    doc.build(story)

    return filename

def clear_chat():
    conversation_history.clear()
    return []

with gr.Blocks() as demo:

    gr.Markdown("# 🧠 The Syntax Surgeon")
    gr.Markdown("AI Powered Code Fixer using IBM Granite")

    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(lines=2, placeholder="Ask something or paste Python code...")

    with gr.Row():
        temperature = gr.Slider(0.1,1.0,value=0.7,label="Temperature")
        max_tokens = gr.Slider(50,1024,value=256,label="Max Tokens")
        top_p = gr.Slider(0.1,1.0,value=0.95,label="Top P")

    send = gr.Button("Send")

    with gr.Row():
        clear = gr.Button("Clear Chat")
        txt = gr.Button("Download TXT")
        pdf = gr.Button("Download PDF")

    def chat(user_message, history, temperature, max_tokens, top_p):

        response = generate_response(
            user_message,
            temperature,
            max_tokens,
            top_p
        )

        history.append((user_message, response))
        return "", history

    send.click(
        chat,
        [msg, chatbot, temperature, max_tokens, top_p],
        [msg, chatbot]
    )

    clear.click(lambda: [], None, chatbot)

demo.launch(share=True)


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded successfully!


/tmp/ipykernel_450/1882788850.py:167: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_450/1882788850.py:167: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://44f3b90fccb2cd0a7e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
# ================================
# The Syntax Surgeon
# AI Code Fixer + Chatbot
# ================================

import re
import ast
import torch
import gradio as gr
import black
import autopep8
from transformers import pipeline
from datetime import datetime
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

# -------------------------------
# Configuration
# -------------------------------

class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95


# -------------------------------
# Load AI Model
# -------------------------------

device = 0 if torch.cuda.is_available() else -1

print("Loading Granite model...")

model_pipe = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME,
    device=device
)

print("Model Loaded Successfully!")


# -------------------------------
# Conversation History
# -------------------------------

conversation_history = []


# -------------------------------
# Detect Python Code
# -------------------------------

def detect_python_code(text):

    patterns = [
        r"def\s+\w+\s*\(",
        r"class\s+\w+",
        r"import\s+\w+",
        r"from\s+\w+\s+import",
        r"if\s+.*:",
        r"for\s+.*\s+in\s+",
        r"while\s+.*:"
    ]

    for pattern in patterns:
        if re.search(pattern, text):
            return True

    return False


# -------------------------------
# Check Syntax
# -------------------------------

def check_syntax(code):

    try:
        ast.parse(code)
        return True, None

    except SyntaxError as e:
        return False, f"Syntax Error (line {e.lineno}): {e.msg}"


# -------------------------------
# Fix Indentation
# -------------------------------

def fix_code_indentation(code):

    lines = code.split("\n")
    fixed_lines = []

    for line in lines:
        line = line.rstrip()
        line = line.replace("\t", "    ")
        fixed_lines.append(line)

    return "\n".join(fixed_lines)


# -------------------------------
# Fix Missing Brackets
# -------------------------------

def fix_code_brackets(code):

    brackets = {"(": ")", "[": "]", "{": "}"}
    stack = []

    for char in code:
        if char in brackets:
            stack.append(brackets[char])
        elif char in brackets.values():
            if stack and stack[-1] == char:
                stack.pop()

    while stack:
        code += stack.pop()

    return code


# -------------------------------
# Format Code
# -------------------------------

def format_code(code):

    try:
        formatted = black.format_str(code, mode=black.FileMode())
        return formatted

    except Exception:
        return autopep8.fix_code(code)


# -------------------------------
# Main Python Code Fixer
# -------------------------------

def fix_python_code(code):

    code = fix_code_indentation(code)
    code = fix_code_brackets(code)

    syntax_ok, error = check_syntax(code)

    if syntax_ok:
        code = format_code(code)
        return "✅ Fixed Python Code:\n\n" + code

    else:
        return f"⚠️ {error}\n\nAttempted Fix:\n{code}"


# -------------------------------
# Generate AI Response
# -------------------------------

def generate_response(message, temperature, max_tokens, top_p):

    if detect_python_code(message):
        return fix_python_code(message)

    conversation_history.append({
        "role": "user",
        "content": message
    })

    result = model_pipe(
        message,
        max_new_tokens=int(max_tokens),
        temperature=temperature,
        do_sample=True,
        top_p=top_p
    )

    response = result[0]["generated_text"]

    conversation_history.append({
        "role": "assistant",
        "content": response
    })

    return response


# -------------------------------
# Export TXT
# -------------------------------

def download_txt():

    filename = "chat_history.txt"

    with open(filename, "w") as f:
        for msg in conversation_history:
            role = msg["role"].upper()
            f.write(f"{role}: {msg['content']}\n\n")

    return filename


# -------------------------------
# Export PDF
# -------------------------------

def download_pdf():

    filename = "chat_history.pdf"

    styles = getSampleStyleSheet()
    story = []

    for msg in conversation_history:
        role = msg["role"].upper()
        text = f"{role}: {msg['content']}"

        story.append(Paragraph(text, styles["Normal"]))
        story.append(Spacer(1, 10))

    doc = SimpleDocTemplate(filename)
    doc.build(story)

    return filename


# -------------------------------
# Clear Chat
# -------------------------------

def clear_chat():
    conversation_history.clear()
    return []


# -------------------------------
# Chat Function
# -------------------------------

def chat(user_message, history, temperature, max_tokens, top_p):

    response = generate_response(
        user_message,
        temperature,
        max_tokens,
        top_p
    )

    history.append((user_message, response))

    return "", history


# -------------------------------
# Gradio Interface
# -------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🧠 The Syntax Surgeon")
    gr.Markdown("AI Powered Python Code Fixer & Chatbot")

    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(
        lines=2,
        placeholder="Ask something or paste Python code..."
    )

    with gr.Row():
        temperature = gr.Slider(0.1, 1.0, value=0.7, label="Temperature")
        max_tokens = gr.Slider(50, 1024, value=256, label="Max Tokens")
        top_p = gr.Slider(0.1, 1.0, value=0.95, label="Top-P")

    send = gr.Button("Send")

    with gr.Row():
        clear = gr.Button("Clear Chat")
        txt = gr.Button("Download TXT")
        pdf = gr.Button("Download PDF")

    send.click(
        chat,
        [msg, chatbot, temperature, max_tokens, top_p],
        [msg, chatbot]
    )

    clear.click(lambda: [], None, chatbot)

demo.launch(share=True)

Loading Granite model...


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Model Loaded Successfully!


/tmp/ipykernel_450/2416391465.py:269: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_450/2416391465.py:269: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1bbd243795d917f9a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
